In this notebook we train two models (model_start and model_end) and then a third model_theta.

In [1]:
from models import CIFAR10ConvNet
import torch
from scheduler import make_diy_scheduler
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from train import train
from curve_model import Curve
import os
import matplotlib.pyplot as plt
import torchmetrics
from curve_plots import plot_Curve_losslandscape, bezier_plot
import pandas as pd

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
torch.manual_seed(0)
root = ".."
datafolder = f"{root}/data"
base_directory = f"{root}/experiments/results_notebook_CIFAR"

In [2]:
MODEL = CIFAR10ConvNet
model_kargs = {"dropout": 0.5}
loss_fn = torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean', label_smoothing=0.0)
batch_size = 256

dataset = "CIFAR10"
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root=f'{datafolder}/train', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root=f'{datafolder}/test', train=False, download=True, transform=transform)
#subset_test = Subset(test_dataset, indices=range(len(test_dataset) // 1))
#subset_train = Subset(train_dataset, indices=range(len(train_dataset) // 1))
test_loader = DataLoader(test_dataset, batch_size=batch_size)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

os.makedirs(f"{base_directory}/models", exist_ok=True)
os.makedirs(f"{base_directory}/figures", exist_ok=True)


/Users/simondanieleiriksson/Documents/GitHub/mode-connectivity/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:

retrain=True
if retrain:
    model_lr_start = torch.tensor(1e-4)
    model_lr_end = torch.tensor(2e-3)
    model_epochs = 100

    total_iter = model_epochs*train_loader.__len__()

    model_start = MODEL(**model_kargs).to(device)
    optimizer_start = torch.optim.Adam(params=model_start.parameters(), lr=model_lr_start.clone())
    scheduler_start = make_diy_scheduler(optimizer_start, 
                                             train_num_steps=total_iter, 
                                             lr_start_warmup=model_lr_start.clone(), 
                                             lr=model_lr_end.clone(), 
                                             lr_warmup_steps=5*train_loader.__len__(), 
                                             lr_finetune_halftime=total_iter // (5*3), 
                                             lr_finetune_steps=total_iter // 3
            )
    model_start, all_train_losses, lrs, epoch_train_losses, test_losses, epoch_train_accuracy, plots = train(model_start, 
                                                train_loader=train_loader, 
                                                test_loader=test_loader, 
                                                optimizer=optimizer_start, 
                                                scheduler=scheduler_start, 
                                                epochs=model_epochs, loss_fn=loss_fn, device=device, 
                                                plot=True, plotpath=f"{base_directory}/start_model/figures",
                                                verbose=True, print_every_n_epoch=5
                                                )
    torch.save(model_start, f"{base_directory}/models/model_start_{MODEL.__name__}_{dataset}.pth")
    for k in plots.keys():
        display(plots[k][0])
else:
    model_start = torch.load(f"{base_directory}/models/model_start_{MODEL.__name__}_{dataset}.pth", map_location=torch.device(device), weights_only=False)

epoch = 5 	train loss: 1.36264, test loss: 1.23696, test accuracy: 56.15, lr: 2.000000e-03


In [ ]:
if retrain:
    model_lr_start = torch.tensor(1e-4)
    model_lr_end = torch.tensor(2e-3)
    model_epochs = 100

    total_iter = model_epochs*train_loader.__len__()

    model_end = MODEL(**model_kargs).to(device)
    optimizer_end = torch.optim.Adam(params=model_end.parameters(), lr=model_lr_start.clone())
    scheduler_end = make_diy_scheduler(optimizer_end, 
                                             train_num_steps=total_iter, 
                                             lr_start_warmup=model_lr_start.clone(), 
                                             lr=model_lr_end.clone(), 
                                             lr_warmup_steps=5*train_loader.__len__(), 
                                             lr_finetune_halftime=total_iter // (5*3), 
                                             lr_finetune_steps=total_iter // 3
            )
    model_end, all_train_losses, lrs, epoch_train_losses, test_losses, epoch_train_accuracy, plots = train(model_end, 
                                                train_loader=train_loader, 
                                                test_loader=test_loader, 
                                                optimizer=optimizer_end, 
                                                scheduler=scheduler_end, 
                                                epochs=model_epochs, loss_fn=loss_fn, device=device, 
                                                plot=True, plotpath=f"{base_directory}/end_model/figures",
                                                verbose=True, print_every_n_epoch=5
                                                )
    torch.save(model_end, f"{base_directory}/models/model_end_{MODEL.__name__}_{dataset}.pth")
    for k in plots.keys():
        display(plots[k][0])
else:
    model_end = torch.load(f"{base_directory}/models/model_end_{MODEL.__name__}_{dataset}.pth", map_location=torch.device(device), weights_only=False)

In [ ]:
def curve_fn(param_start, param_end, param_theta, t):
    return param_start * (1-t)**2 + param_end * t**2 + param_theta * 2*t*(1-t)

In [ ]:
retrain_curve = True
curve_epochs= 200
curve_lr_start = torch.tensor(1e-4)
curve_lr_end = torch.tensor(1e-1)
curve_optimizer = "SGD"
total_iter = curve_epochs*train_loader.__len__()


if retrain_curve:
    total_iter = curve_epochs*train_loader.__len__()
    curve = Curve(model_start=model_start, model_end=model_end, curve_fn=curve_fn, device=device)
    gamma = ((curve_lr_end.log()-curve_lr_start.log())/(curve_epochs*train_loader.__len__())).exp()

    optimizer = torch.optim.SGD(params=curve.model_theta.parameters(), lr=curve_lr_start, momentum=0.9)

    scheduler = make_diy_scheduler(optimizer, 
                                            train_num_steps=total_iter, 
                                            lr_start_warmup=curve_lr_start.clone(), 
                                            lr=curve_lr_end.clone(), 
                                            lr_warmup_steps=5*train_loader.__len__(), 
                                            lr_finetune_halftime=total_iter // (5*3), 
                                            lr_finetune_steps=total_iter // 3
        )
            

    curve, all_train_losses, lrs, epoch_train_losses, test_losses, epoch_train_accuracy, plots = train(curve, 
                                                                        train_loader=train_loader, 
                                                                        test_loader=test_loader, 
                                                                        optimizer=optimizer, 
                                                                        scheduler=scheduler, 
                                                                        epochs=curve_epochs,
                                                                        loss_fn=loss_fn, 
                                                                        device=device, 
                                                                        plot=True, 
                                                                        plotpath=f"{base_directory}/curve_model/figures", 
                                                                        #plotname=f"curvefitting_{MODEL.__name__}_{dataset}", 
                                                                        modeltype="curve", 
                                                                        verbose=True, print_every_n_epoch=5)
    torch.save(curve.model_theta, f"{base_directory}/models/curve.model_theta_{MODEL.__name__}_{dataset}.pth")
    for k in plots.keys():
        display(plots[k][0])
else:
    curve = Curve(model_start=model_start, model_end=model_end, curve_fn=curve_fn, device=device)
    curve.model_theta = torch.load(f"{base_directory}/models/curve.model_theta_{MODEL.__name__}_{dataset}.pth", map_location=torch.device(device), weights_only=False)
    

In [ ]:
# fig, ax = plot_Curve_losslandscape(curve, device, f"{base_directory}/figures", test_loader, N_points=10, loss_fn=loss_fn, recalc_mesh=True, N_bezierpoints=30)
# fig.savefig(f"{base_directory}/figures/loss_landscape.png")
# plt.show(fig)

In [ ]:
metrics_dict = {
    "Cross Entropy": lambda pred_probs, target: torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean', label_smoothing=0.0)(pred_probs.log(), target),
    "Expected Calibration Error": torchmetrics.classification.MulticlassCalibrationError(num_classes=10, n_bins=25).to(device),
    "Accuracy": lambda pred_probs, target: torchmetrics.functional.classification.accuracy(preds=pred_probs, target=target, task="multiclass", num_classes=10).to(device),
    "AUROC": lambda pred_probs, target: torchmetrics.functional.classification.auroc(preds=pred_probs, target=target.to(torch.long), task="multiclass", num_classes=10).to(device)
}


In [ ]:
fig, axs, eval_results = bezier_plot(curve, device, test_loader=test_loader, 
                                     plottype="linear", 
                    N_bezierpoints = 30,
                    plot_linear=False, metrics_dict=metrics_dict)
fig.savefig(f"{base_directory}/figures/metric_along_curve.png")
plt.show(fig)

In [ ]:
print(pd.DataFrame(eval_results["curve_ensemble_score_dict"]).T.reset_index().rename(columns={"index": "Model"}).to_markdown(index=False))